In [ ]:
import asyncio
import lsst_efd_client

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pytz
import sys
from astropy.stats import median_absolute_deviation as mad_astropy
from scipy.stats import binned_statistic
from astropy.time import Time

from lsst.summit.utils import (
    ConsDbClient,
    getAirmassSeeingCorrection,
    getBandpassSeeingCorrection,
)
import os

%matplotlib inline

In [ ]:
os.environ["no_proxy"] += ",.consdb"
consdb_url = 'http://consdb-pq.consdb:8080/consdb'
cdb_client = ConsDbClient(consdb_url)

In [ ]:

query = f"""
SELECT
e.obs_start_mjd,
e.exposure_id,
e.band,
q.visit_id,
e.s_ra AS ra,
e.s_dec AS dec,
q.sky_bg_median,
q.guider_e1_mean,
q.guider_e2_mean,
q.guider_altitude_drift,
q.guider_altitude_rms_detrended,
q.guider_altitude_standard_deviation,
q.guider_azimuth_drift,
q.guider_azimuth_rms_detrended,
q.guider_azimuth_standard_deviation,
q.psf_ixx_median,
q.psf_iyy_median,
q.psf_ixy_median
FROM
cdb_lsstcam.visit1_quicklook AS q,
cdb_lsstcam.exposure AS e
WHERE
q.visit_id = e.exposure_id
AND e.img_type = 'science'
AND e.airmass > 0
AND e.band != 'none'
"""

table = cdb_client.query(query).to_pandas()
#table.to_csv('visits_rubin.csv')

In [ ]:
table['T_median'] = table['psf_ixx_median'] + table['psf_iyy_median']
table['e1_median'] = (table['psf_ixx_median'] - table['psf_iyy_median']) / table['T_median']
table['e2_median'] = 2*table['psf_ixy_median'] / table['T_median']
key = table['T_median']
key = np.array(key).astype(float)

# Convert PSF sigma to FWHM
sig2fwhm = 2 * np.sqrt(2 * np.log(2))
pixel_scale = 0.2  # arcsec / pixel
fwhm = np.sqrt(key/2) * sig2fwhm * pixel_scale



In [ ]:
_ = plt.hist(fwhm, bins=np.linspace(0., 3, 100))